# UEMR chốt, file 02 §5: encode Omni `best` các clip mới của dev HT-Step

Theo `docs/uemr_chot/UEMR_CHOT_02_HTSTEP_DEV.md` §4–5. Các dòng R2, R3, R6, R8 của UniAV mới (bộ đoạn đã chọn ở §3, commit trong
`results/uemr_final/dev/segments/`) có clip chưa nằm trong cache lượt dev 1 (`GraceFeel/encode_omni`, `htstep_uemr/a4ca43db/best`). Notebook
này **chỉ encode các clip đó** bằng đúng đường của cột (b) (`omni_retrieval.encode`, mp4 + `timestamps`, như lượt dev 1), ghi store lên Drive và
upload `GraceFeel/encode_omni` (`uemr_final/dev/best/clips/`). Text không encode lại. Việc chấm (§6–8) chạy ở local bằng
`python -m uemr_htstep.final_dev score` (numpy, không GPU).

- Colab Secrets: `HF_TOKEN` (`GraceFeel`). GPU: A100. Đĩa `/content` trống ≥ 60 GB.
- **Chạy thử trước:** `SMOKE = True` (5 video, ghi `smoke/`, không upload). Xong đổi `SMOKE = False`, Run all. Encode resume theo store trên Drive.

In [ ]:
# ===== 0. Cấu hình =====
FUSHION_REF = '143e708f6c279e2c34abb5782966b446c1659c3e'   # commit trên nhánh htstep (user push trước khi chạy)
SMOKE       = True
SEC_REF     = 0.32       # s/clip Omni best trên A100 (lượt dev 1: 0,314)
HF_OUT      = 'GraceFeel/encode_omni'
OUT_PATH    = 'uemr_final/dev/best/clips'
PROTO1_SHA16 = 'a4ca43db9829842a'
OMNI_FIX_REF = 'ab772eb'
FLAGS = {k: v for k, v in dict(globals()).items() if k in ('FUSHION_REF', 'SMOKE', 'HF_OUT', 'OUT_PATH')}

In [ ]:
# ===== 1. Drive, code (Fushion, Omni-fix), thư viện =====
from google.colab import drive, userdata
drive.mount('/content/drive')
import os, sys, json, glob, shutil, subprocess, time, hashlib, platform, csv
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['HF_XET_CHUNK_CACHE_SIZE_BYTES'] = '0'
os.environ.pop('HF_XET_HIGH_PERFORMANCE', None)
T_START = time.time()
FUSHION, OMNI_FIX = '/content/code/Fushion_UEMR', '/content/code/Omni-fix'
os.makedirs('/content/code', exist_ok=True)

def clone(url, path, ref, *branch):
    os.chdir('/content')
    shutil.rmtree(path, ignore_errors=True)
    subprocess.run(['git', 'clone', '-q', *branch, url, path], check=True)
    r = subprocess.run(['git', '-C', path, 'checkout', '-q', ref], capture_output=True, text=True)
    assert r.returncode == 0, f'không checkout được {ref} của {url}: {r.stderr} (đã push chưa?)'
    return subprocess.run(['git', '-C', path, 'log', '-1', '--format=%h %s'], capture_output=True, text=True).stdout.strip()

COMMIT = clone('https://github.com/chien24/Fushion_UEMR.git', FUSHION, FUSHION_REF)
OMNI_COMMIT = clone('https://github.com/im-xiaoming/Omni-Retriever-Custom.git', OMNI_FIX, OMNI_FIX_REF, '-b', 'htstep_lora')
print('Fushion :', COMMIT); print('Omni-fix:', OMNI_COMMIT)
if FUSHION not in sys.path:
    sys.path.insert(0, FUSHION)
os.environ['PYTHONPATH'] = FUSHION + ':' + os.environ.get('PYTHONPATH', '')
os.chdir(FUSHION)

def sh(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = (r.stdout + r.stderr)[-3000:]
    if out.strip():
        print(out)
    if check and r.returncode:
        raise RuntimeError(f'lệnh lỗi ({r.returncode}): {cmd[:200]}')
    return r

def check(name, ok, detail=''):
    print(f"CHECK {name}: {'OK' if ok else 'FAIL'}" + (f' | {detail}' if detail else ''))
    assert ok, f'CHECK {name} FAIL: {detail}'

def sha256(path, bs=1 << 24):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(bs), b''):
            h.update(chunk)
    return h.hexdigest()

def wj(path, obj):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path + '.tmp', 'w', encoding='utf-8') as f:
        json.dump(obj, f, indent=1, ensure_ascii=False, default=str)
    os.replace(path + '.tmp', path)

sh(f'pip install -q -r {FUSHION}/requirements.txt')
sh('pip uninstall -y -q torchao', check=False)
import numpy as np, pandas as pd, torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'không có'
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 2**30 if torch.cuda.is_available() else 0.0
ENV = {'python': platform.python_version(), 'torch': torch.__version__, 'gpu': gpu, 'fushion': COMMIT, 'omni_fix': OMNI_COMMIT, 'smoke': SMOKE}
print(json.dumps(ENV, ensure_ascii=False))
DRIVE = '/content/drive/MyDrive'
H = f'{DRIVE}/Colab Notebooks/code KL/Fushion/uemr_chot_run/dev' + ('/smoke' if SMOKE else '')
LOCAL = '/content/smoke' if SMOKE else '/content/full'
A, VID = '/content/hts', f'{LOCAL}/videos_val'
if SMOKE:
    shutil.rmtree(H, ignore_errors=True); shutil.rmtree(LOCAL, ignore_errors=True)
for d in (H, A, VID, '/content/videos'):
    os.makedirs(d, exist_ok=True)

In [ ]:
# ===== 2. Adapter best, cache dev 1, danh sách clip mới (theo khoá video__ts_te) =====
from huggingface_hub import hf_hub_download, HfApi
from omni_retrieval.manifest import segment_table, unique_segments, segment_records, store_keys, write_jsonl
from uemr_htstep import data as hd
PR = json.load(open(f'{FUSHION}/configs/htstep_uemr_protocol.json', encoding='utf-8'))
check('protocol lượt dev 1 (nguồn hash)', sha256(f'{FUSHION}/configs/htstep_uemr_protocol.json')[:16] == PROTO1_SHA16)
D, M = PR['data'], PR['models']
BEST_ADAPTER = '/content/adapters/best'
if not os.path.exists(f'{BEST_ADAPTER}/adapter_model.safetensors'):
    sh("hf download GraceFeel/omni_lora_htstep --include 'lora_bs32/adapter_best/*' --local-dir /content/adapters/_hf")
    shutil.copytree('/content/adapters/_hf/lora_bs32/adapter_best', BEST_ADAPTER, dirs_exist_ok=True)
check('sha256 adapter best = 5e9899aa…', sha256(f'{BEST_ADAPTER}/adapter_model.safetensors') == M['omni_best']['sha256'])
db1 = f'htstep_uemr/{PROTO1_SHA16[:8]}/best/db'
for f in ('segments.npz', 'meta.json'):
    hf_hub_download('GraceFeel/encode_omni', f'{db1}/{f}', repo_type='dataset', local_dir=A)
META1 = json.load(open(f'{A}/{db1}/meta.json'))
check('cache dev 1: sha256 segments.npz = meta.json', sha256(f'{A}/{db1}/segments.npz') == META1['sha256']['segments.npz'])
with np.load(f'{A}/{db1}/segments.npz') as z:
    CACHE = set(z['seg_key'].astype(str))
SEGD = f'{FUSHION}/results/uemr_final/dev/segments'
SEL = json.load(open(f'{SEGD}/selected.json', encoding='utf-8'))
for n, h in SEL['files'].items():
    check(f'sha256 {n}.json = selected.json', sha256(f'{SEGD}/{n}.json') == h)
ROWS = json.load(open(f'{SEGD}/rows.json', encoding='utf-8'))
for n in ('r3', 'r6'):
    check(f'sha256 {n}.json = rows.json', sha256(f'{SEGD}/{n}.json') == ROWS['files'][n])
PARTS = {n: json.load(open(f'{SEGD}/{n}.json', encoding='utf-8')) for n in ('r2', 'r3', 'r6', 'r8')}
VIDS = sorted(PARTS['r8'])
check('kho 510 video', VIDS == sorted(D['gallery']), str(len(VIDS)))
if SMOKE:
    VIDS = VIDS[:5]
    PARTS = {n: {v: p[v] for v in VIDS} for n, p in PARTS.items()}
UNIQ = unique_segments(segment_table(PARTS))
NEW = {k: u for k, u in UNIQ.items() if k not in CACHE}
by_row = {n: sum(1 for v in p for s in p[v] if f'{v}__{s[0]:.2f}_{s[1]:.2f}' not in CACHE) for n, p in PARTS.items()}
print(f'θ {SEL["theta"]}, L_min {SEL["l_min"]}, L_fill {SEL["l_fill"]}, L (R2) {SEL["chunk_sec"]} s')
print(f'clip khác nhau: {len(UNIQ)} | đã có trong cache dev 1: {len(UNIQ) - len(NEW)} | cần encode: {len(NEW)} '
      f'(theo dòng, có trùng: {by_row}) | ước lượng {len(NEW) * SEC_REF / 3600:.2f} giờ A100')
INV = {'unique': len(UNIQ), 'new': len(NEW), 'by_row': by_row, 'selected': {k: SEL[k] for k in ('theta', 'l_min', 'l_fill', 'chunk_sec')}}

In [ ]:
# ===== 3. mp4 val (như lượt dev 1, ô 5): tải, AV1 chuyển H.264 không mất mát, thời lượng so với protocol =====
import importlib.util as _ilu
_spec = _ilu.spec_from_file_location('htstep_io', f'{OMNI_FIX}/src/omniretriever/data/htstep_io.py')
hio = _ilu.module_from_spec(_spec); _spec.loader.exec_module(hio)
t0 = time.time()
items_p = hf_hub_download(D['items_val']['hf'], D['items_val']['file'], repo_type='dataset', local_dir=A)
check('sha256 items_val', sha256(items_p) == D['items_val']['sha256'])
SRC = {r['video_id']: r['source'] for r in hd.read_jsonl(items_p)}
TAR = {r['video_id']: (int(r['offset_data']), int(r['size'])) for r in csv.DictReader(open(f'{OMNI_FIX}/data/htstep/tar_index.csv', newline='', encoding='utf-8'))}
check('sha256 tar_index.csv', sha256(f'{OMNI_FIX}/data/htstep/tar_index.csv') == D['tar_index']['sha256'])
NEED_V = sorted({u['video_id'] for u in NEW.values()})

def fetch_one(v):
    dst = f'{VID}/{v}.mp4'
    if os.path.exists(dst) and os.path.getsize(dst):
        return v, 'có sẵn'
    tmp = f'/content/tmp_vid/{v}'
    os.makedirs(tmp, exist_ok=True)
    try:
        s, raw = SRC[v], f'{tmp}/raw.mp4'
        if 'tar' in s:
            hio.range_get(s['tar'], *TAR[v], raw, repo=s['repo'])
        else:
            hio.hf_get(s['repo'], s['file'], raw)
        shutil.copy2(raw, dst + '.part')
        os.replace(dst + '.part', dst)
        return v, 'tải'
    except Exception as e:     # noqa: BLE001 - một video lỗi không làm mất các video khác
        return v, f'lỗi: {type(e).__name__}: {e}'
    finally:
        shutil.rmtree(tmp, ignore_errors=True)

def to_local(v):
    src, dst = f'{VID}/{v}.mp4', f'/content/videos/{v}.mp4'
    if os.path.exists(dst) and os.path.getsize(dst):
        return v, 'có sẵn'
    p = hio.probe(src)
    if p['codec'] != 'av1':
        shutil.copy2(src, dst + '.part'); os.replace(dst + '.part', dst)
        return v, 'copy'
    ok, why, _ = hio.transcode(src, dst + '.h264.mp4', p)
    if not ok:
        return v, f'lỗi transcode: {why}'
    os.replace(dst + '.h264.mp4', dst)
    return v, 'av1 -> h264'

def measure(v):
    p = hio.probe(f'/content/videos/{v}.mp4')
    num, den = (int(x) for x in (p['fps'] or '0/1').split('/'))
    return v, p['frames'] * den / num if p['frames'] and num else p['duration']

free = shutil.disk_usage('/content').free / 2**30
check('đĩa /content đủ (>= 60 GB trống)', free >= 60, f'{free:.0f} GB')
with ThreadPoolExecutor(4) as ex:
    STATUS = dict(ex.map(fetch_one, NEED_V))
bad = {v: s for v, s in STATUS.items() if s.startswith('lỗi')}
check('mọi video mp4 có đủ', not bad, str(list(bad.items())[:3]))
with ThreadPoolExecutor(8) as ex:
    TO_LOCAL = dict(ex.map(to_local, NEED_V))
bad = {v: s for v, s in TO_LOCAL.items() if s.startswith('lỗi')}
check('mp4 local đọc được bằng decord (AV1 đã chuyển H.264)', not bad, str(list(bad.items())[:3]))
with ThreadPoolExecutor(8) as ex:
    MEAS = dict(ex.map(measure, NEED_V))
gap = {v: abs(MEAS[v] - float(D['durations'][v])) for v in NEED_V}
check('thời lượng mp4 = protocol (lệch <= 0,1 s)', max(gap.values()) <= 0.1, str(sorted(gap.items(), key=lambda x: -x[1])[:3]))
print(f'{len(NEED_V)} video, {dict(Counter(s.split(":")[0] for s in TO_LOCAL.values()))} | {time.time() - t0:.0f} s')

In [ ]:
# ===== 4. Encode (đường cột (b) của lượt dev 1: omni_retrieval.encode, mp4 + timestamps, loader vendored), resume theo store =====
BASE_MODEL, BEATS = '/content/WAVE_HOME/WAVE-7B', '/content/WAVE_HOME/BEATs_iter3_plus_AS2M_finetuned_on_AS2M_cpt2.pt'
STORE = f'{H}/best/clips'

def wave_ok():
    idx = glob.glob(f'{BASE_MODEL}/*.index.json')
    return os.path.exists(BEATS) and bool(idx) and all(os.path.exists(f'{BASE_MODEL}/{s}') for s in set(json.load(open(idx[0]))['weight_map'].values()))

check('GPU đủ cho Omni 7B (>= 22 GB)', GPU_GB >= 22, f'{gpu}, {GPU_GB:.0f} GB')
chk = subprocess.run([sys.executable, '-c', 'import decord, av, peft, torch, transformers; print(transformers.__version__, torch.cuda.is_available())'], capture_output=True, text=True)
last = (chk.stdout.strip().splitlines() or [''])[-1]
check('môi trường encode (transformers 4.51.3, CUDA)', chk.returncode == 0 and last.split()[:2] == ['4.51.3', 'True'], f'{last} {chk.stderr[-300:]}')
if not wave_ok():
    sh('hf download nguyenminh04/omni-model --local-dir /content/WAVE_HOME --max-workers 2')
check('WAVE-7B + BEATs đủ file', wave_ok())
ENC = f'cd {FUSHION} && python -m omni_retrieval.encode --base-model {BASE_MODEL} --beats-path {BEATS} --adapter {BEST_ADAPTER} --video-root /content/videos'

def encode(manifest, extra=''):
    with open(f'{H}/encode.log', 'a') as f:
        f.write(f"\n===== {time.strftime('%Y-%m-%d %H:%M:%S')} =====\n")
    get_ipython().system(f'{ENC} --modality av --manifest "{manifest}" --store "{STORE}" {extra} 2>&1 | tee -a "{H}/encode.log"')

t0 = time.time()
todo = segment_records(NEW, store_keys(STORE, 'av'))
print(f'cần encode lần này: {len(todo)} / {len(NEW)} | ước lượng {len(todo) * SEC_REF / 3600:.2f} giờ')
if todo:
    write_jsonl(f'{H}/segs_todo.jsonl', todo)
    if len(todo) > 20:
        encode(f'{H}/segs_todo.jsonl', '--num-workers 4 --limit 20')
        spc = json.loads(open(f'{STORE}/runs.jsonl').read().strip().splitlines()[-1]).get('sec_per_clip') or float('nan')
        print(f'20 clip: {spc:.3f} s/clip -> còn {len(todo) - 20} clip ≈ {(len(todo) - 20) * spc / 3600:.2f} giờ')
    encode(f'{H}/segs_todo.jsonl', '--num-workers 4')
left = [r['id'] for r in segment_records(NEW, store_keys(STORE, 'av'))]
err = {}
if os.path.exists(f'{STORE}/failed.jsonl'):
    for l in open(f'{STORE}/failed.jsonl'):
        if l.strip():
            q = json.loads(l); err[q['id']] = q['error']
pd.DataFrame([{'seg_key': k, 'error': err.get(k, 'không có trong failed.jsonl')} for k in left], columns=['seg_key', 'error']).to_csv(f'{H}/failed.csv', index=False)
fail_pct = 100 * len(left) / max(len(UNIQ), 1)
INV.update(failed=len(left), failed_pct=fail_pct, sec_encode=round(time.time() - t0, 1))
check('clip lỗi <= 2 % (file 02 §5)', fail_pct <= 2, f'{len(left)} clip ({fail_pct:.2f} %), xem {H}/failed.csv')

In [ ]:
# ===== 5. Upload store, tóm tắt cho Nhật ký file 02 =====
runs = [json.loads(l) for l in open(f'{STORE}/runs.jsonl')] if os.path.exists(f'{STORE}/runs.jsonl') else []
INV.update(sec_per_clip=[r.get('sec_per_clip') for r in runs], chunks=len(glob.glob(f'{STORE}/chunk_*.npz')), env=ENV, flags=FLAGS,
           sec_total=round(time.time() - T_START, 1))
wj(f'{H}/inventory.json', INV)
if not SMOKE:
    api = HfApi()
    api.upload_folder(repo_id=HF_OUT, repo_type='dataset', folder_path=STORE, path_in_repo=OUT_PATH, allow_patterns=['chunk_*.npz', 'failed.jsonl', 'runs.jsonl', 'encoder_meta.json'],
                      commit_message=f'UEMR chốt dev: clip mới Omni best (Fushion {COMMIT[:7]})')
    print('đã upload:', f'{HF_OUT}/{OUT_PATH}')
print(json.dumps({k: v for k, v in INV.items() if k not in ('env', 'flags')}, indent=1, ensure_ascii=False))
print('Gửi agent: output ô 2, 3, 4, 5. Agent chấm ở local (final_dev score) rồi khoá configs/uemr_final.json.')
if SMOKE:
    print('SMOKE xong: mọi CHECK OK. Đổi SMOKE = False ở ô 0 rồi Run all.')